# 006-lora-baseline: Full Evaluation

Evaluate the Qwen2.5-1.5B LoRA adapter on PAN2025 validation and 5 out-of-distribution datasets.

**Datasets**:
- PAN2025 Validation (in-distribution)
- HC3 Wiki (OOD)
- RAID (OOD)
- TuringBench (OOD)
- M4 (OOD)
- Ghostbuster (OOD)

**Requirements**: Run on a GPU runtime (T4 or better).

In [13]:
# =====================================================
# CELL 0 — Environment setup
# =====================================================
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA L4


In [14]:
# =====================================================
# CELL 1 — Install dependencies
# =====================================================
# Pin datasets<3.0 to avoid 'Dataset scripts are no longer supported' error
!pip install -q peft gdown scikit-learn 'datasets<3.0'

In [ ]:
# =====================================================
# CELL 2 — Authenticate with HuggingFace
# =====================================================
from huggingface_hub import login

# Option 1: Read token from Colab secrets (recommended)
try:
    from google.colab import userdata
    login(token="hf_xxx")
    print('Authenticated.')
except Exception:
    # Option 2: Interactive login
    from huggingface_hub import notebook_login
    print('Colab secret not found. Please log in manually:')
    notebook_login()

Authenticated.


In [19]:
# =====================================================
# CELL 3 — Download all datasets
# =====================================================
import json, random, requests
import gdown
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
random.seed(42)

def save_jsonl(records, path):
    with open(path, 'w') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
    print(f'  Saved {len(records)} records to {path.name}')

# ─── PAN2025 Validation (Google Drive) ────────────────
print('Downloading PAN2025 val...')
val_id = '12szC1TcNPN9KULPNjTZsEyG8RafxfsJy'
gdown.download(
    f'https://drive.google.com/uc?id={val_id}',
    str(DATA_DIR / 'val.jsonl'), quiet=True
)
print('  Done.')

# ─── HC3 Wiki (direct JSONL download) ─────────────────
print('\nDownloading HC3 (wiki subset)...')
hc3_url = 'https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/wiki_csai.jsonl'
resp = requests.get(hc3_url)
resp.raise_for_status()
records = []
for line in resp.text.strip().split('\n'):
    row = json.loads(line)
    for ans in row.get('human_answers', []):
        if ans.strip():
            records.append({'text': ans.strip(), 'label': 0, 'source': 'hc3_human'})
    for ans in row.get('chatgpt_answers', []):
        if ans.strip():
            records.append({'text': ans.strip(), 'label': 1, 'source': 'hc3_chatgpt'})
save_jsonl(records, DATA_DIR / 'hc3_wiki.jsonl')

# ─── RAID (streaming to avoid huge download) ──────────
print('\nDownloading RAID (streaming)...')
raid = load_dataset('liamdugan/raid', 'raid', split='train', streaming=True)
human_recs, ai_recs = [], []
for row in raid:
    if row.get('language', 'en') != 'en':
        continue
    text = row.get('generation', '')
    if not text or len(text.strip()) < 50:
        continue
    label = 0 if row.get('model') == 'human' else 1
    rec = {'text': text.strip(), 'label': label, 'source': 'raid'}
    if label == 0:
        human_recs.append(rec)
    else:
        ai_recs.append(rec)
    # Stop early once we have enough
    if len(human_recs) >= 1500 and len(ai_recs) >= 1500:
        break
n = min(1000, len(human_recs), len(ai_recs))
records = random.sample(human_recs, n) + random.sample(ai_recs, n)
random.shuffle(records)
save_jsonl(records, DATA_DIR / 'raid.jsonl')

# ─── Summary ──────────────────────────────────────────
print('\n=== Dataset Summary ===')
for f in sorted(DATA_DIR.glob('*.jsonl')):
    with open(f) as fh:
        lines = fh.readlines()
    labels = [json.loads(l)['label'] for l in lines]
    n0 = labels.count(0)
    n1 = labels.count(1)
    print(f'  {f.name:30s}  {len(lines):>6d} samples  (human={n0}, ai={n1})')

  Done.

  Saved 1684 records to hc3_wiki.jsonl

  Saved 2000 records to raid.jsonl

=== Dataset Summary ===
  hc3_wiki.jsonl                    1684 samples  (human=842, ai=842)
  raid.jsonl                        2000 samples  (human=1000, ai=1000)
  train.jsonl                      23707 samples  (human=9101, ai=14606)
  val.jsonl                         3589 samples  (human=1277, ai=2312)


In [20]:
# =====================================================
# CELL 4 — Load model + LoRA adapter
# =====================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from peft import PeftModel

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
ADAPTER_REPO = 'hersheys-baklava/qwen-lora-pan2026'

print(f'Loading base model: {MODEL_NAME}')
config = AutoConfig.from_pretrained(MODEL_NAME)
config.pad_token_id = config.eos_token_id
config.num_labels = 2
config.problem_type = 'single_label_classification'

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=config, torch_dtype=torch.bfloat16,
)

print(f'Loading LoRA adapter from: {ADAPTER_REPO}')
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model = model.merge_and_unload()
model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

print(f'Model loaded on {device}. Params: {sum(p.numel() for p in model.parameters()):,}')

Loading base model: Qwen/Qwen2.5-1.5B


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading LoRA adapter from: hersheys-baklava/qwen-lora-pan2026


adapter_config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/8.74M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.43k [00:00<?, ?B/s]

Model loaded on cuda. Params: 1,543,717,376


In [21]:
# =====================================================
# CELL 5 — Utilities: chunking, metrics, evaluation
# =====================================================
import numpy as np
from sklearn.metrics import roc_auc_score, brier_score_loss, f1_score, fbeta_score
from tqdm import tqdm
import pandas as pd

CHUNK_SIZE = 512
CHUNK_STRIDE = 256


def chunk_text_by_tokens(input_ids, attention_mask, chunk_size=512,
                         stride=256, pad_token_id=0):
    if len(input_ids) <= chunk_size:
        return [input_ids], [attention_mask]
    chunked_ids, chunked_masks = [], []
    start = 0
    while start < len(input_ids):
        end = min(start + chunk_size, len(input_ids))
        c_ids = input_ids[start:end]
        c_mask = attention_mask[start:end]
        if len(c_ids) < chunk_size:
            pad_len = chunk_size - len(c_ids)
            c_ids = c_ids + [pad_token_id] * pad_len
            c_mask = c_mask + [0] * pad_len
        chunked_ids.append(c_ids)
        chunked_masks.append(c_mask)
        start += stride
        if start + stride >= len(input_ids):
            break
    return chunked_ids, chunked_masks


def get_all_metrics(y_true, y_pred, y_prob):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)
    n = len(y_true)
    # C@1
    unanswered = (y_prob == 0.5)
    n_u = np.sum(unanswered)
    n_c = np.sum((y_pred == y_true) & (~unanswered))
    c_at_1 = (1/n) * (n_c + n_u * (n_c / n)) if n > 0 else 0
    # F0.5u
    mod_pred = y_pred.copy()
    mod_pred[unanswered] = 0
    return {
        'ROC-AUC': float(roc_auc_score(y_true, y_prob)),
        'Brier': float(brier_score_loss(y_true, y_prob)),
        'F1': float(f1_score(y_true, y_pred)),
        'C@1': float(c_at_1),
        'F0.5u': float(fbeta_score(y_true, mod_pred, beta=0.5)),
    }


def predict_one(model, tokenizer, text, device):
    enc = tokenizer(text, truncation=False, return_tensors='pt',
                   add_special_tokens=True)
    ids = enc['input_ids'][0].tolist()
    mask = enc['attention_mask'][0].tolist()
    c_ids, c_masks = chunk_text_by_tokens(
        ids, mask, chunk_size=CHUNK_SIZE, stride=CHUNK_STRIDE,
        pad_token_id=tokenizer.pad_token_id,
    )
    probs = []
    with torch.no_grad():
        for ci, cm in zip(c_ids, c_masks):
            inp = {
                'input_ids': torch.tensor([ci]).to(device),
                'attention_mask': torch.tensor([cm]).to(device),
            }
            out = model(**inp)
            p = torch.softmax(out.logits.float(), dim=1)
            probs.append(p[0, 1].item())
    return float(np.mean(probs))


def evaluate_dataset(model, tokenizer, path, device, max_samples=None):
    with open(path) as f:
        df = pd.DataFrame([json.loads(l) for l in f])
    if max_samples:
        df = df.head(max_samples)
    y_true = df['label'].tolist()
    y_probs = []
    for text in tqdm(df['text'], desc=f'  {Path(path).stem}', leave=True):
        y_probs.append(predict_one(model, tokenizer, text, device))
    y_probs = np.array(y_probs)
    y_pred = (y_probs >= 0.5).astype(int)
    metrics = get_all_metrics(y_true, y_pred, y_probs)
    return metrics, y_probs.tolist()


print('Utilities loaded.')

Utilities loaded.


In [22]:
# =====================================================
# CELL 6 — Evaluate on all datasets
# =====================================================
from datetime import datetime

DATASETS = [
    ('PAN2025 Validation', DATA_DIR / 'val.jsonl',          'pan2025_val'),
    ('HC3 Wiki',           DATA_DIR / 'hc3_wiki.jsonl',     'hc3_wiki'),
    ('RAID',               DATA_DIR / 'raid.jsonl',         'raid'),
]

all_results = {}
all_predictions = {}

print(f'Started: {datetime.now().isoformat()}')
print(f'Device: {device}\n')

for name, path, key in DATASETS:
    if not path.exists():
        print(f'\n  Warning: {name} not found at {path}, skipping.')
        continue
    print(f'\n{"─" * 60}')
    print(f'Dataset: {name}')
    print(f'{"─" * 60}')
    metrics, probs = evaluate_dataset(model, tokenizer, str(path), device)
    all_results[name] = metrics
    all_predictions[key] = probs
    print(f'\n  Results:')
    for k, v in metrics.items():
        print(f'    {k}: {v:.4f}')

print(f'\nCompleted: {datetime.now().isoformat()}')

Started: 2026-02-13T05:29:00.698433
Device: cuda


────────────────────────────────────────────────────────────
Dataset: PAN2025 Validation
────────────────────────────────────────────────────────────


  val: 100%|██████████| 3589/3589 [06:08<00:00,  9.74it/s]



  Results:
    ROC-AUC: 0.9999
    Brier: 0.0019
    F1: 0.9981
    C@1: 0.9975
    F0.5u: 0.9984

────────────────────────────────────────────────────────────
Dataset: HC3 Wiki
────────────────────────────────────────────────────────────


  hc3_wiki: 100%|██████████| 1684/1684 [01:03<00:00, 26.41it/s]



  Results:
    ROC-AUC: 0.9982
    Brier: 0.0112
    F1: 0.9868
    C@1: 0.9869
    F0.5u: 0.9947

────────────────────────────────────────────────────────────
Dataset: RAID
────────────────────────────────────────────────────────────


  raid: 100%|██████████| 2000/2000 [01:14<00:00, 27.02it/s]


  Results:
    ROC-AUC: 0.8152
    Brier: 0.1491
    F1: 0.8182
    C@1: 0.8460
    F0.5u: 0.9176

Completed: 2026-02-13T05:37:27.271921


In [23]:
# =====================================================
# CELL 7 — Comparison table
# =====================================================
baselines = {
    'LR (001)':         {'PAN2025': 0.9953, 'HC3': 0.5517, 'RAID': 0.5727,
                         'TuringBench': 0.5449, 'M4': 0.5409, 'Ghostbuster': 0.5227},
    'LightGBM (002)':   {'PAN2025': 0.9980, 'HC3': 0.3356, 'RAID': 0.3398,
                         'TuringBench': 0.3209, 'M4': 0.3170, 'Ghostbuster': 0.3012},
    'DeBERTa FT (003)': {'PAN2025': 0.9948, 'HC3': 0.5939, 'RAID': 0.5879,
                         'TuringBench': 0.5660, 'M4': 0.5591, 'Ghostbuster': 0.5889},
}

key_map = {
    'PAN2025 Validation': 'PAN2025', 'HC3 Wiki': 'HC3', 'RAID': 'RAID',
    'TuringBench': 'TuringBench', 'M4': 'M4', 'Ghostbuster': 'Ghostbuster',
}
datasets_short = ['PAN2025', 'HC3', 'RAID']

qwen_scores = {key_map.get(n, n): m.get('ROC-AUC', 0) for n, m in all_results.items()}

print('=' * 110)
print('ROC-AUC Comparison: All Models × All Datasets')
print('=' * 110)
header = f'{"Model":<20}' + ''.join(f'{d:>14}' for d in datasets_short) + f'{"Avg OOD":>14}'
print(header)
print('-' * len(header))

for model_name, scores in baselines.items():
    ood = [scores.get(d, 0) for d in datasets_short[1:]]
    row = f'{model_name:<20}'
    for d in datasets_short:
        row += f'{scores.get(d, 0):>14.4f}'
    row += f'{np.mean(ood):>14.4f}'
    print(row)

ood = [qwen_scores.get(d, 0) for d in datasets_short[1:]]
row = f'{"Qwen LoRA (006)":<20}'
for d in datasets_short:
    row += f'{qwen_scores.get(d, 0):>14.4f}'
row += f'{np.mean(ood):>14.4f}'
print(row)
print('=' * len(header))

# Full metrics
print(f'\n\nFull Metrics (Qwen LoRA 006):')
print(f'{"Dataset":<20} {"ROC-AUC":>10} {"Brier":>10} {"F1":>10} {"C@1":>10} {"F0.5u":>10}')
print('-' * 70)
for name, metrics in all_results.items():
    print(f'{name:<20} {metrics["ROC-AUC"]:>10.4f} {metrics["Brier"]:>10.4f} '
          f'{metrics["F1"]:>10.4f} {metrics["C@1"]:>10.4f} {metrics["F0.5u"]:>10.4f}')

ROC-AUC Comparison: All Models × All Datasets
Model                      PAN2025           HC3          RAID       Avg OOD
----------------------------------------------------------------------------
LR (001)                    0.9953        0.5517        0.5727        0.5622
LightGBM (002)              0.9980        0.3356        0.3398        0.3377
DeBERTa FT (003)            0.9948        0.5939        0.5879        0.5909
Qwen LoRA (006)             0.9999        0.9982        0.8152        0.9067


Full Metrics (Qwen LoRA 006):
Dataset                 ROC-AUC      Brier         F1        C@1      F0.5u
----------------------------------------------------------------------
PAN2025 Validation       0.9999     0.0019     0.9981     0.9975     0.9984
HC3 Wiki                 0.9982     0.0112     0.9868     0.9869     0.9947
RAID                     0.8152     0.1491     0.8182     0.8460     0.9176


In [24]:
# =====================================================
# CELL 8 — Upload results to HuggingFace adapter repo
# =====================================================
from huggingface_hub import HfApi
import tempfile

results_payload = {
    'model': 'Qwen/Qwen2.5-1.5B + LoRA',
    'adapter_repo': ADAPTER_REPO,
    'evaluated_at': datetime.now().isoformat(),
    'device': device,
    'chunk_size': CHUNK_SIZE,
    'chunk_stride': CHUNK_STRIDE,
    'metrics': all_results,
    'predictions': all_predictions,
}

with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json.dump(results_payload, f, indent=2)
    tmp_path = f.name

api = HfApi()
api.upload_file(
    path_or_fileobj=tmp_path,
    path_in_repo='evaluation_results.json',
    repo_id=ADAPTER_REPO,
    commit_message='Upload full evaluation results (PAN2025 + 5 OOD datasets)',
)

os.remove(tmp_path)
print(f'Results uploaded to https://huggingface.co/{ADAPTER_REPO}')
print(f'File: evaluation_results.json')

Results uploaded to https://huggingface.co/hersheys-baklava/qwen-lora-pan2026
File: evaluation_results.json
